In [1]:
from pathlib import Path
import pandas as pd

In [2]:
RAW_FILE = Path(
    r"F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer"
    r"\review-analyzer\data\raw\swiggy_reviews_ipynb.csv"
)

In [3]:

CLEAN_DIR = Path(
    r"F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer"
    r"\review-analyzer\data\clean"
)

In [4]:
print("Raw file exists:", RAW_FILE.exists())
print("Clean directory exists:", CLEAN_DIR.exists())

Raw file exists: True
Clean directory exists: True


In [5]:
df_raw = pd.read_csv(RAW_FILE)

In [6]:
print("\nRaw dataset loaded successfully.")
print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
print("\nColumns:")
print(df_raw.columns.tolist())


Raw dataset loaded successfully.
Rows: 5000
Columns: 11

Columns:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']


In [7]:
missing = df_raw.isna().sum()
missing_percent = (missing / len(df_raw) * 100).round(2)

missing_report = pd.DataFrame({
    "Missing Values": missing,
    "Missing %": missing_percent
})

print(missing_report)

                      Missing Values  Missing %
reviewId                           0        0.0
userName                           0        0.0
userImage                          0        0.0
content                            0        0.0
score                              0        0.0
thumbsUpCount                      0        0.0
reviewCreatedVersion             525       10.5
at                                 0        0.0
replyContent                       5        0.1
repliedAt                          5        0.1
appVersion                       525       10.5


In [8]:
# Check duplicate rows
duplicate_rows = df_raw.duplicated().sum()

# Check duplicate review IDs
duplicate_review_ids = df_raw["reviewId"].duplicated().sum()

# Check duplicate review text
duplicate_content = (
    df_raw["content"]
    .astype(str)
    .str.strip()
    .duplicated()
    .sum()
)

print("Duplicate rows:", duplicate_rows)
print("Duplicate review IDs:", duplicate_review_ids)
print("Duplicate review texts:", duplicate_content)

Duplicate rows: 0
Duplicate review IDs: 0
Duplicate review texts: 2071


In [9]:
duplicate_text_counts = (
    df_raw["content"]
    .astype(str)
    .str.strip()
    .value_counts()
)

print("Unique review texts:", len(duplicate_text_counts))

print("\nMost repeated review texts:")
print(duplicate_text_counts.head(20))

Unique review texts: 2929

Most repeated review texts:
content
good            712
nice            215
good 👍          109
super           100
ok               79
very good        73
Good             65
best             51
good service     49
excellent        41
👍                40
very nice        34
good app         28
good 😊           17
nice 👍           17
great            17
Nice             16
nice app         15
awesome          15
gud              13
Name: count, dtype: int64


In [10]:
empty_reviews = (
    df_raw["content"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Empty or whitespace-only reviews:", empty_reviews)

Empty or whitespace-only reviews: 0


In [11]:
from langdetect import detect, LangDetectException

def detect_language(text):
    try:
        text = str(text).strip()
        if not text:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"

df_raw["detected_language"] = df_raw["content"].apply(detect_language)

print("Language distribution:")
print(df_raw["detected_language"].value_counts())

Language distribution:
detected_language
en         2142
so         1073
pl          292
af          286
ro          186
unknown     130
sk           84
ca           80
hr           75
it           75
da           73
id           65
cs           59
fr           57
cy           43
no           38
tl           33
et           32
nl           25
de           21
sl           19
sv           17
sw           16
hi           12
es           10
tr            9
sq            9
fi            9
pt            7
hu            6
vi            5
ta            3
bn            2
mr            2
te            2
ml            1
kn            1
lv            1
Name: count, dtype: int64


In [12]:
for lang in ["so", "pl", "af", "ro"]:
    print(f"\n--- {lang} examples ---")
    print(
        df_raw.loc[
            df_raw["detected_language"] == lang,
            "content"
        ].head(10).to_string(index=False)
    )


--- so examples ---
  good
  good
  good
  good
  Good
  good
  Good
good 👍
Good 👍
  good

--- pl examples ---
     nice 👍
nice swiggy
       nice
       nice
       nice
       nice
       nice
       nice
       nice
       nice

--- af examples ---
     awesome
        best
       worst
        best
        best
   very good
   very good
very good 😊💯
          Ok
  awesome 👌🏼

--- ro examples ---
      super
      great
      super
     Nice 👍
super super
      super
    great 👍
      super
use ful app
     Nice 👍


In [13]:
# Show examples from some longer reviews detected as non-English

for lang in ["so", "pl", "af", "ro"]:
    examples = df_raw[
        (df_raw["detected_language"] == lang) &
        (df_raw["content"].astype(str).str.len() > 50)
    ]["content"].head(5)

    print(f"\n--- {lang} longer examples ---")
    print(examples.to_string(index=False))


--- so longer examples ---
free wala item nhi diye... select karne baad bh...
buzz offer band kar diya bina kesi wajaha say j...

--- pl longer examples ---
Series([], )

--- af longer examples ---
order karo but anandam se nahi kyo ki wo wrong ...
bakwaas!!! service bekaar hai sab kuch bekar ha...
very very very very very very good service very...

--- ro longer examples ---
Series([], )


In [14]:
import re

non_latin = df_raw["content"].astype(str).str.contains(
    r"[\u0900-\u097F\u0980-\u09FF\u0A00-\u0A7F\u0B00-\u0B7F\u0C00-\u0C7F\u0D00-\u0D7F]",
    regex=True
)

print("Reviews containing Indian-language scripts:", non_latin.sum())

Reviews containing Indian-language scripts: 26


In [15]:
import re

text = df_raw["content"].astype(str)

urls = text.str.contains(
    r"https?://|www\.",
    regex=True,
    case=False
)

emails = text.str.contains(
    r"\b[\w.-]+@[\w.-]+\.\w+\b",
    regex=True
)

html_tags = text.str.contains(
    r"<[^>]+>",
    regex=True
)

print("Reviews with URLs:", urls.sum())
print("Reviews with email addresses:", emails.sum())
print("Reviews with HTML-like tags:", html_tags.sum())

Reviews with URLs: 0
Reviews with email addresses: 0
Reviews with HTML-like tags: 0


In [16]:
text = df_raw["content"].astype(str)

leading_spaces = text.str.match(r"^\s").sum()
trailing_spaces = text.str.match(r"\s$").sum()
multiple_spaces = text.str.contains(r"\s{2,}", regex=True).sum()

print("Reviews with leading whitespace:", leading_spaces)
print("Reviews with trailing whitespace:", trailing_spaces)
print("Reviews with multiple consecutive spaces:", multiple_spaces)

Reviews with leading whitespace: 0
Reviews with trailing whitespace: 0
Reviews with multiple consecutive spaces: 2


In [17]:
uppercase_reviews = (
    df_raw["content"]
    .astype(str)
    .str.contains(r"[A-Z]", regex=True)
    .sum()
)

print("Reviews containing uppercase letters:", uppercase_reviews)

Reviews containing uppercase letters: 1306


In [18]:
import string

punctuation_reviews = df_raw["content"].astype(str).apply(
    lambda x: any(char in string.punctuation for char in x)
).sum()

print("Reviews containing punctuation:", punctuation_reviews)

Reviews containing punctuation: 1237


In [19]:
review_lengths = df_raw["content"].astype(str).str.len()

print("Average characters:", round(review_lengths.mean(), 2))
print("Shortest review:", review_lengths.min())
print("Longest review:", review_lengths.max())
print("Reviews with 10 characters or less:", (review_lengths <= 10).sum())
print("Reviews over 500 characters:", (review_lengths > 500).sum())

Average characters: 59.58
Shortest review: 1
Longest review: 500
Reviews with 10 characters or less: 2381
Reviews over 500 characters: 0


In [20]:
short_reviews = (
    df_raw[df_raw["content"].astype(str).str.len() <= 10]["content"]
    .astype(str)
    .str.strip()
)

print("Short reviews:", len(short_reviews))
print("\nMost common short reviews:")
print(short_reviews.value_counts().head(30))

Short reviews: 2381

Most common short reviews:
content
good         712
nice         215
good 👍       109
super        100
ok            79
very good     73
Good          65
best          51
excellent     41
👍             40
very nice     34
good app      28
nice 👍        17
good 😊        17
great         17
Nice          16
nice app      15
awesome       15
gud           13
thank you     11
wow           10
worst         10
okay           8
superb         8
good👍          8
worst app      7
good 👍🏻        7
👌              7
nice 🙂         7
good 👍😊        7
Name: count, dtype: int64


In [21]:
print("Unique scores:")
print(sorted(df_raw["score"].unique()))

print("\nInvalid scores:")
invalid_scores = df_raw[
    ~df_raw["score"].isin([1, 2, 3, 4, 5])
]

print("Count:", len(invalid_scores))


Unique scores:
[1, 2, 3, 4, 5]

Invalid scores:
Count: 0


In [22]:
print("Data type:", df_raw["thumbsUpCount"].dtype)
print("Minimum value:", df_raw["thumbsUpCount"].min())
print("Negative values:", (df_raw["thumbsUpCount"] < 0).sum())

Data type: int64
Minimum value: 0
Negative values: 0


In [23]:
dates = pd.to_datetime(df_raw["at"], errors="coerce")

print("Missing/invalid dates:", dates.isna().sum())
print("Earliest review date:", dates.min())
print("Latest review date:", dates.max())

Missing/invalid dates: 0
Earliest review date: 2026-08-03 18:38:06
Latest review date: 2026-08-14 23:55:43


In [24]:
reply_content_missing = df_raw["replyContent"].isna()
reply_date_missing = df_raw["repliedAt"].isna()

print("Reply content missing:", reply_content_missing.sum())
print("Reply date missing:", reply_date_missing.sum())

print(
    "Reply content present but reply date missing:",
    ((~reply_content_missing) & reply_date_missing).sum()
)

print(
    "Reply date present but reply content missing:",
    (reply_content_missing & (~reply_date_missing)).sum()
)

Reply content missing: 5
Reply date missing: 5
Reply content present but reply date missing: 0
Reply date present but reply content missing: 0


In [25]:
print("reviewCreatedVersion type:", df_raw["reviewCreatedVersion"].dtype)
print("appVersion type:", df_raw["appVersion"].dtype)

print("\nSample reviewCreatedVersion:")
print(df_raw["reviewCreatedVersion"].dropna().head(10).to_string(index=False))

print("\nSample appVersion:")
print(df_raw["appVersion"].dropna().head(10).to_string(index=False))

reviewCreatedVersion type: object
appVersion type: object

Sample reviewCreatedVersion:
4.114.2
4.114.2
4.113.4
4.114.2
4.114.2
4.114.2
4.114.2
4.113.4
4.114.2
4.114.2

Sample appVersion:
4.114.2
4.114.2
4.113.4
4.114.2
4.114.2
4.114.2
4.114.2
4.113.4
4.114.2
4.114.2


In [26]:
print("reviewId data type:", df_raw["reviewId"].dtype)
print("Empty review IDs:", (
    df_raw["reviewId"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
))

print("\nSample review IDs:")
print(df_raw["reviewId"].head(10).to_string(index=False))

reviewId data type: object
Empty review IDs: 0

Sample review IDs:
6666e860-ad46-4a18-80dd-b05cd936d80c
64931981-3256-4303-a12c-620f0d95d052
25777dcf-9734-4bcb-b03f-f9dc1c65af07
f05df8a7-0133-48ab-a4f5-51768a031a83
3e48f28c-b21f-45c5-8d83-2fb72c5c7786
69ad7255-0f06-4edd-a559-62a4a5d05107
7fdfda2a-b35b-4b27-9568-ae5e87d09501
ab1006de-9358-49a9-961d-d093761f6352
23ee3934-dda2-467e-bba2-874e7829a70b
0d5a8735-21cd-4d01-8c4b-cd80994367d4


In [27]:
print("Missing userName:", df_raw["userName"].isna().sum())

print(
    "Empty userName:",
    df_raw["userName"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nSample usernames:")
print(df_raw["userName"].head(10).to_string(index=False))

Missing userName: 0
Empty userName: 0

Sample usernames:
A Google user
A Google user
A Google user
A Google user
A Google user
A Google user
A Google user
A Google user
A Google user
A Google user


In [28]:
print("Missing userImage:", df_raw["userImage"].isna().sum())

print(
    "Empty userImage:",
    df_raw["userImage"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nSample userImage values:")
print(df_raw["userImage"].head(5).to_string(index=False))

Missing userImage: 0
Empty userImage: 0

Sample userImage values:
https://play-lh.googleusercontent.com/EGemoI2NT...
https://play-lh.googleusercontent.com/EGemoI2NT...
https://play-lh.googleusercontent.com/EGemoI2NT...
https://play-lh.googleusercontent.com/EGemoI2NT...
https://play-lh.googleusercontent.com/EGemoI2NT...


In [29]:
print("Missing appVersion:", df_raw["appVersion"].isna().sum())

print("\nUnique app versions:")
print(df_raw["appVersion"].value_counts().head(20))

Missing appVersion: 525

Unique app versions:
appVersion
4.113.4    2590
4.114.2     986
4.112.0     239
4.113.1     206
4.111.1      92
4.109.1      53
4.110.1      35
4.114.1      33
4.113.3      20
4.108.0      16
4.104.0      16
4.108.1      12
4.107.2       9
4.106.1       9
4.101.2       8
4.114.0       7
4.91.1        6
4.101.0       6
4.103.2       6
4.99.1        6
Name: count, dtype: int64


In [30]:
print("Missing reviewCreatedVersion:", df_raw["reviewCreatedVersion"].isna().sum())

print("\nUnique reviewCreatedVersion values:")
print(df_raw["reviewCreatedVersion"].value_counts().head(20))

Missing reviewCreatedVersion: 525

Unique reviewCreatedVersion values:
reviewCreatedVersion
4.113.4    2590
4.114.2     986
4.112.0     239
4.113.1     206
4.111.1      92
4.109.1      53
4.110.1      35
4.114.1      33
4.113.3      20
4.108.0      16
4.104.0      16
4.108.1      12
4.107.2       9
4.106.1       9
4.101.2       8
4.114.0       7
4.91.1        6
4.101.0       6
4.103.2       6
4.99.1        6
Name: count, dtype: int64


In [31]:
no_reply = df_raw[
    df_raw["replyContent"].isna() &
    df_raw["repliedAt"].isna()
]

print("Reviews without a reply:", len(no_reply))

print("\nReview IDs:")
print(no_reply["reviewId"].to_string(index=False))

print("\nReview content:")
print(no_reply["content"].to_string(index=False))

Reviews without a reply: 5

Review IDs:
2cf9e795-7709-4d67-acf8-f01a677063ec
64aeef36-19dd-4312-8048-7bd050323f22
d0245bf1-42ec-43c9-b9ab-4ad8bd834caf
25aa0b79-5b21-4b95-bb73-4b70e589626b
d885e9ae-15d4-4bbc-89a0-271e5a9d852b

Review content:
                                           Good 😊👍
taking more than order gst,taxes, delivery char...
                                           super 🥰
                                         excellent
              It is a scam app cheating customers.


In [32]:
normalized_text = (
    df_raw["content"]
    .astype(str)
    .str.strip()
    .str.lower()
)

long_duplicate_counts = (
    normalized_text[normalized_text.str.len() > 30]
    .value_counts()
)

print("Unique long review texts:", len(long_duplicate_counts))

print("\nMost repeated long reviews:")
print(long_duplicate_counts.head(20))

Unique long review texts: 1571

Most repeated long reviews:
content
quality and quantity is good but price is high.                                                                                                                                                                                                                                                                                                                                                                                                                                                                         1
i am not able to track my order, i have reported app crashes multiple times and it isn't fixed. every time i order food i have to wait till the delivery partner contacts me! i can't even open the order page on the app.                                                                                                                                                                                                                

In [33]:
df_clean = df_raw.copy()

print("Raw rows:", len(df_raw))
print("Working rows:", len(df_clean))

Raw rows: 5000
Working rows: 5000


In [34]:
df_clean["content"] = (
    df_clean["content"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Rows after text normalization:", len(df_clean))
print("\nSample cleaned reviews:")
print(df_clean["content"].head(10).to_string(index=False))

Rows after text normalization: 5000

Sample cleaned reviews:
   quality and quantity is good but price is high.
                                           awesome
  i don't able to use my coupan code after 4 order
very bad and disappointing experience with swig...
muje nhi pta ki aap logo k paas ab kya dilivery...
                      fast order and good delivery
overpriced and delivering no quality food total...
                                    super delivery
                                              good
                       best food app good delivery


In [35]:
print("Leading whitespace:",
      df_clean["content"].str.match(r"^\s").sum())

print("Trailing whitespace:",
      df_clean["content"].str.match(r"\s$").sum())

print("Multiple spaces:",
      df_clean["content"].str.contains(r"\s{2,}", regex=True).sum())

print("Uppercase letters:",
      df_clean["content"].str.contains(r"[A-Z]", regex=True).sum())

Leading whitespace: 0
Trailing whitespace: 0
Multiple spaces: 0
Uppercase letters: 0


In [36]:
empty_after_cleaning = (
    df_clean["content"]
    .isna()
    |
    df_clean["content"].str.strip().eq("")
).sum()

print("Empty reviews after cleaning:", empty_after_cleaning)

Empty reviews after cleaning: 0


In [37]:
duplicates = df_clean.duplicated().sum()

print("Duplicate rows after cleaning:", duplicates)

Duplicate rows after cleaning: 0


In [38]:
duplicate_ids = df_clean["reviewId"].duplicated().sum()

print("Duplicate review IDs after cleaning:", duplicate_ids)

Duplicate review IDs after cleaning: 0


In [39]:
print("Sample cleaned reviews:\n")

print(
    df_clean[["content", "score"]]
    .head(15)
    .to_string(index=False)
)

Sample cleaned reviews:

                                                                                                                                                                                                                                                                                                                                                                                                                      content  score
                                                                                                                                                                                                                                                                                                                                                                              quality and quantity is good but price is high.      5
                                                                                                                                     

In [40]:
from pathlib import Path

CLEAN_FILE = Path(
    r"F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer"
    r"\review-analyzer\data\clean\swiggy_reviews_clean_ipynb.csv"
)

df_clean.to_csv(CLEAN_FILE, index=False, encoding="utf-8")

print("Cleaned dataset saved to:")
print(CLEAN_FILE)
print("\nRows saved:", len(df_clean))

Cleaned dataset saved to:
F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer\review-analyzer\data\clean\swiggy_reviews_clean_ipynb.csv

Rows saved: 5000


In [41]:
print("Raw shape:", df_raw.shape)
print("Clean shape:", df_clean.shape)

print("\nColumns unchanged:", list(df_raw.columns) == list(df_clean.columns))

print(
    "Review IDs unchanged:",
    df_raw["reviewId"].equals(df_clean["reviewId"])
)

print(
    "Scores unchanged:",
    df_raw["score"].equals(df_clean["score"])
)

Raw shape: (5000, 12)
Clean shape: (5000, 12)

Columns unchanged: True
Review IDs unchanged: True
Scores unchanged: True
